In [1]:
import time
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torchode
from pathlib import Path
import pickle

from scipy.integrate import solve_ivp as sp_solve_ivp
from ftnode.data import TrialsDataset
from tqdm.auto import tqdm

from sklearn.preprocessing import MinMaxScaler
from ftnode.utils import set_global_seed, _load_loop_wrapper
from ftnode.data import TrialsDataset
from ftnode.node import (
    FTNODE, FeluSigmoidMLP,FeluSigmoidMLPfeaturized,
     GeluSigmoidMLPfeaturized,
)

import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams['font.family']= 'serif'

device = 'cpu'
seed = 1234
set_global_seed(seed=seed)
random_state = 67

[Seed] Deterministic mode enabled (may reduce speed).


In [2]:
def hysteresis_ode(t,x,lam):
    return lam+x-x**3

In [3]:
n_lam = 51
n_traj = 51
lams = np.linspace(-1,1,n_lam)
xs = np.linspace(-2,2,n_traj)


In [4]:
t_max = 0.25
n_colloc = 101


Xs = []
Us = []
t = np.linspace(0,t_max,n_colloc)
for lami in tqdm(lams):
    for x0 in xs:
        sol = sp_solve_ivp(
            hysteresis_ode,
            t_span = [0,t_max],
            y0 = np.array(x0).reshape(-1),
            t_eval = np.linspace(0,t_max,n_colloc),
            args = (lami,)
        )

        Xs.append(sol.y.T)
        Us.append([lami])
Xs = np.array(Xs)
Us = np.array(Us)

  0%|          | 0/51 [00:00<?, ?it/s]

In [5]:
dXs = np.zeros_like(Xs)
T = t[np.newaxis,:,np.newaxis]
X_diff = Xs[:,2:,:] - Xs[:,:-2,:]
T_diff = T[:,2:,:] - T[:,:-2,:]

dXs[:,1:-1,:] = X_diff/T_diff
dXs[:,0,:] = (Xs[:,1,:] - Xs[:,0,:]) / (T[:,1,:] - T[:,0,:])
dXs[:,-1,:] = (Xs[:,-1,:] - Xs[:,-2,:]) / (T[:,-1,:] - T[:,-2,:])

In [6]:
scaler = MinMaxScaler(feature_range=(-1,1))
Xs_scaled = scaler.fit_transform(Xs.reshape(-1,1).reshape(-1,1)).reshape(-1,n_colloc,1)

In [7]:
dX_tensor = [
    torch.tensor(dxi,dtype=torch.float32,device=device) for dxi in dXs
]
X_tensor = [
    torch.tensor(xi,dtype=torch.float32,device=device) for xi in Xs_scaled
]
U_tensor = [
    torch.tensor(ui,dtype=torch.float32,device=device) for ui in Us
]

T_tensor = [
    torch.tensor(t,dtype=torch.float32, device=device) for _ in range(len(Xs))
]

In [8]:
class GradDataset(torch.utils.data.Dataset):
    def __init__(self, dX: List, X: List, T: List, U: List):
        self.dX = dX
        self.X = X
        self.T = T
        self.U = U
        # self.trans_idx = Transient_idx

    def __len__(self):
        return len(self.dX)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError(
                f"Index {idx} is out of bounds of dataset size: {len(self)}."
            )

        dXi = self.dX[idx]
        Xi = self.X[idx]
        ti = self.T[idx]
        ui = self.U[idx]

        return dXi, Xi, ti, ui

dataset = GradDataset(dX = dX_tensor,X = X_tensor, T = T_tensor, U = U_tensor)

# Run $k$-Folds Cross-validation

In [9]:
k_folds = 10

In [10]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, SubsetRandomSampler
import time
import copy

In [11]:
avg_best_val_losses = []
avg_best_train_losses = []

In [12]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 200  
batch_size = 50 
learning_rate = 1e-2
print_every = 10
_precision = 6
random_state = random_state
solve_method = 'tsit5'


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)

val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(
        dims=[1,10,10,10,10, 1],
        activation=nn.SiLU(),
        lower_bound=-4,
        upper_bound=-0.1,
        init_type=None
    )


    g = GeluSigmoidMLPfeaturized(
        dims=[6, 10,10,10,1],
        activation=nn.SiLU(),
        lower_bound=-2,
        upper_bound=2,
        freq_sample_step=1,
        feat_lower_bound=-1.5,
        feat_upper_bound=1.5,
        init_type=None
    )

    model = FTNODE(f, g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    fold_seed = random_state + fold
    set_global_seed(fold_seed)

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            x0i = Xi[:, 0, :]
            
            # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            # u_func = lambda t: ui_expanded
            u_func = lambda t: ui
            func = lambda t, x: model_fold(t, x, u_func)

            opt.zero_grad()
            sol = torchode.solve_ivp(
                f=func,
                y0=x0i,
                t_eval=ti.squeeze(),
                method=solve_method,
            )
            
            Xi_pred = sol.ys


            # loss = loss_criteria(dXi, dXi_pred)
            loss = loss_criteria(Xi, Xi_pred)
                
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                x0i = Xi[:, 0, :]
                # ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                # u_func = lambda t: ui_expanded
                u_func = lambda t: ui
                func = lambda t, x: model_fold(t, x, u_func)

                sol = torchode.solve_ivp(
                    f=func,
                    y0=x0i,
                    t_eval=ti.squeeze(),
                    method=solve_method,
                )

                Xi_pred = sol.ys


                # loss = loss_criteria(dXi, dXi_pred)
                loss = loss_criteria(Xi, Xi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---
[Seed] Deterministic mode enabled (may reduce speed).


Fold 1:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.507702e-03, Val Loss = 1.943743e-03, Time = 3.955877e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 1.105981e-03, Val Loss = 4.831144e-04, Time = 4.190674e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 2.847667e-04, Val Loss = 1.962967e-04, Time = 4.173550e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.578376e-04, Val Loss = 1.206683e-04, Time = 4.167131e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 1.058884e-04, Val Loss = 8.490445e-05, Time = 3.976785e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 7.260082e-05, Val Loss = 6.506218e-05, Time = 3.784521e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.568273e-05, Val Loss = 1.051932e-05, Time = 3.833104e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 5.692333e-06, Val Loss = 6.012966e-06, Time = 3.730171e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 4.772134e-06, Val Loss = 2.783590e-06, Time = 3.711016e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 2.533831e-06, Val Loss = 2.686826e-06, Time = 3.555681e+00, lr = 1.000000e

Fold 2:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.441474e-03, Val Loss = 2.106409e-03, Time = 3.109374e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 1.037272e-03, Val Loss = 4.013231e-04, Time = 3.740845e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 2.435365e-04, Val Loss = 1.265624e-04, Time = 3.870599e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 1.003435e-04, Val Loss = 7.217484e-05, Time = 3.713817e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 6.082799e-05, Val Loss = 5.719151e-05, Time = 3.731862e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 4.711419e-05, Val Loss = 3.941451e-05, Time = 3.751614e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.687174e-05, Val Loss = 2.069141e-05, Time = 3.680273e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 2.711617e-06, Val Loss = 3.188397e-06, Time = 3.638655e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 2.094572e-06, Val Loss = 1.563882e-06, Time = 3.423608e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 1.237771e-06, Val Loss = 5.950582e-07, Time = 3.394909e+00, lr = 1.000000e

...

Fold 9:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.427979e-03, Val Loss = 1.858633e-03, Time = 3.096265e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 1.108730e-03, Val Loss = 6.428998e-04, Time = 3.630133e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 4.940065e-04, Val Loss = 4.038919e-04, Time = 3.789571e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 3.007086e-04, Val Loss = 1.885980e-04, Time = 3.667915e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 1.619128e-04, Val Loss = 1.200229e-04, Time = 3.552117e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 1.002284e-04, Val Loss = 6.903528e-05, Time = 3.563900e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 1.200923e-05, Val Loss = 8.804183e-06, Time = 3.568773e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 3.466507e-06, Val Loss = 3.396000e-06, Time = 3.466035e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 2.973739e-06, Val Loss = 1.894808e-06, Time = 3.378251e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 8.855657e-07, Val Loss = 7.415860e-07, Time = 3.303669e+00, lr = 5.000000e

Fold 10:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 0: Train Loss = 4.303350e-03, Val Loss = 1.622251e-03, Time = 2.976633e+00, lr = 1.000000e-02
Epoch 1: Train Loss = 7.605907e-04, Val Loss = 1.816492e-04, Time = 3.088208e+00, lr = 1.000000e-02
Epoch 2: Train Loss = 1.217920e-04, Val Loss = 9.738021e-05, Time = 3.449334e+00, lr = 1.000000e-02
Epoch 3: Train Loss = 8.015028e-05, Val Loss = 6.496264e-05, Time = 3.511133e+00, lr = 1.000000e-02
Epoch 4: Train Loss = 5.896914e-05, Val Loss = 7.383375e-05, Time = 3.474079e+00, lr = 1.000000e-02
Epoch 5: Train Loss = 5.053791e-05, Val Loss = 4.894166e-05, Time = 3.416914e+00, lr = 1.000000e-02
Epoch 10: Train Loss = 2.357671e-05, Val Loss = 2.429751e-05, Time = 3.320508e+00, lr = 1.000000e-02
Epoch 20: Train Loss = 1.020690e-05, Val Loss = 1.039558e-05, Time = 3.172320e+00, lr = 1.000000e-02
Epoch 30: Train Loss = 5.265223e-06, Val Loss = 6.767488e-06, Time = 3.134428e+00, lr = 1.000000e-02
Epoch 40: Train Loss = 3.481227e-06, Val Loss = 2.536207e-06, Time = 2.987271e+00, lr = 1.000000e

([1.222111414437658e-07], 0)